data Preprocessing

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = 224
BATCH_SIZE = 32

datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    horizontal_flip=True,
    zoom_range=0.2,
    validation_split=0.2
)

train_data = datagen.flow_from_directory(
    "dataset/",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training'
)

val_data = datagen.flow_from_directory(
    "dataset/",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation'
)

Transfer learning with MobileNetv2

In [5]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation="relu")(x)
output = Dense(1, activation="sigmoid")(x)

mobilenet_model = Model(inputs=base_model.input, outputs=output)

for layer in base_model.layers:
    layer.trainable = False

mobilenet_model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

Model Train

In [ ]:
mobilenet_model.fit(
    train_data,
    validation_data=val_data,
    epochs=5
)

mobilenet_model.save("models/mobilenet_model.h5")

Transfer learning with EfficientNet

In [7]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

base_model = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation="relu")(x)
output = Dense(1, activation="sigmoid")(x)

efficientnet_model = Model(inputs=base_model.input, outputs=output)

efficientnet_model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

EfficientNet training

In [ ]:
efficientnet_model.fit(
    train_data,
    validation_data=val_data,
    epochs=5
)

efficientnet_model.save("models/efficientnet_model.h5")

Grad CAM

In [17]:
import tensorflow as tf
import numpy as np
import cv2

def generate_gradcam(model, img_array, layer_name):

    grad_model = tf.keras.models.Model(
        [model.inputs],
        [model.get_layer(layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:

        conv_outputs, predictions = grad_model(img_array)
        loss = predictions[:,0]

    grads = tape.gradient(loss, conv_outputs)

    pooled_grads = tf.reduce_mean(grads, axis=(0,1,2))

    conv_outputs = conv_outputs[0]

    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = np.maximum(heatmap,0) / np.max(heatmap)

    return heatmap

In [18]:
def overlay_heatmap(heatmap, image_path):

    img = cv2.imread(image_path)

    heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))

    heatmap = np.uint8(255 * heatmap)

    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

    superimposed_img = heatmap * 0.4 + img

    return superimposed_img

Ensemble learning (combines both prediction)

In [ ]:
import numpy as np
import cv2
from tensorflow.keras.models import load_model

mobilenet = load_model("models/mobilenet_model.h5")
efficientnet = load_model("models/efficientnet_model.h5")

IMG_SIZE = 224

def predict(image_path):

    img = cv2.imread(image_path)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img / 255.0

    img = np.expand_dims(img, axis=0)

    pred1 = mobilenet.predict(img)
    pred2 = efficientnet.predict(img)

    final_pred = (pred1 + pred2) / 2

    if final_pred > 0.5:
        return "Fake"
    else:
        return "Real"

Grad cam

In [25]:
import tensorflow as tf
import numpy as np
import cv2

def generate_gradcam(model, img_array, layer_name):

    grad_model = tf.keras.models.Model(
        [model.inputs],
        [model.get_layer(layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:

        conv_outputs, predictions = grad_model(img_array)
        loss = predictions[:,0]

    grads = tape.gradient(loss, conv_outputs)

    pooled_grads = tf.reduce_mean(grads, axis=(0,1,2))

    conv_outputs = conv_outputs[0]

    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = np.maximum(heatmap,0) / np.max(heatmap)

    return heatmap

Gradio interface

In [ ]:
import numpy as np
import cv2
import gradio as gr
from tensorflow.keras.models import load_model

mobilenet = load_model("models/mobilenet_model.h5")
efficientnet = load_model("models/efficientnet_model.h5")

IMG_SIZE = 224

def predict(image_path):

    img = cv2.imread(image_path)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img / 255.0
    img = np.expand_dims(img, axis=0)

    pred1 = mobilenet.predict(img)
    pred2 = efficientnet.predict(img)

    final_pred = (pred1 + pred2) / 2

    confidence = float(final_pred[0][0])

    label = "Fake" if confidence > 0.5 else "Real"

    return label, round(confidence*100,2)

interface = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="filepath"),
    outputs=["text","number"],
    title="Deepfake Detection System"
)

interface.launch()